# NHANES to Vivarium Risk Exposure Demo

This project demonstrates how to extract survey data from NHANES,
summarize it into GBD-compatible age/sex bins, build a vivarium data artifact,
and initialize a microsimulation with the resulting risk factor exposure.

## Example: Liver Stiffness Measurement (LSM)

NHANES began collecting liver stiffness via FibroScan (vibration-controlled
transient elastography) in the 2017-2018 cycle. Higher LSM values indicate
increased liver fibrosis. We model it two ways: as a continuous risk factor
(lognormal/ensemble distribution over kPa) and as a categorical risk factor
(fibrosis stages F0--F4).

## Notebooks

| # | Notebook | Purpose |
|---|----------|---------|
| 01 | `01_nhanes_to_vivarium.ipynb` | Continuous risk: download NHANES LSM data, build artifact, run simulation |
| 02 | `02_transformation_experiments.ipynb` | Test sqrt/log transforms to improve continuous distribution fits |
| 03 | `03_categorical_risk.ipynb` | Categorical risk: fibrosis stages F0--F4 as ordered polytomous exposure |

### Sections in Notebook 01

1. **Data download** -- NHANES demographics + liver stiffness from 2 cycles
2. **Analysis** -- weighted mean and SD by GBD 5-year age groups and sex
3. **Artifact creation** -- build vivarium HDF5 artifact with lognormal risk exposure
4. **Simulation** -- 10,000 simulants with Q-Q validation against NHANES
5. **Trajectories** -- track individual simulant exposures over ~5 years
6. **Bootstrap uncertainty** -- 10 parameter draws showing trajectory uncertainty
7. **Ensemble distribution** -- AIC-weighted mixture of distribution types

### Sections in Notebook 02

1. **Skewness visualization** -- how sqrt/log transforms affect LSM distribution shape
2. **Systematic experiment** -- 30 combinations of transform x distribution x fitting method
3. **Fibrosis staging comparison** -- which combination best matches NHANES F3/F4 prevalence
4. **Q-Q plots** -- top candidates compared to NHANES by age/sex strata
5. **Per-stratum validation** -- check fit consistency across age/sex bins
6. **Vivarium integration** -- end-to-end test with log transform + lognormal in vivarium

### Sections in Notebook 03

1. **Fibrosis prevalence** -- compute weighted F0--F4 fractions by age/sex from NHANES
2. **Categorical artifact** -- build ordered polytomous risk factor with 5 categories
3. **Simulation** -- 10,000 simulants assigned to fibrosis stages
4. **Validation** -- per-stratum comparison to NHANES
5. **Head-to-head** -- categorical vs. continuous fibrosis staging accuracy
6. **Trajectories** -- track stage transitions as simulants age
7. **Trade-offs** -- when to use continuous vs. categorical

## Data Sources

| Cycle | Label | n (adults with LSM) |
|-------|-------|---------------------|
| 2017-March 2020 Pre-Pandemic | `P_` prefix | ~8,300 |
| 2021-2022 | `_L` suffix | ~5,900 |

## Setup

This project requires vivarium and vivarium_public_health. Install into a
conda environment or venv:

```bash
cd nhanes_vivarium_risk_demo
uv venv && uv pip install -r requirements.txt
```

## Key Results

**Notebook 01** (continuous): The lognormal distribution overestimates F4
cirrhosis prevalence (~6% vs. NHANES 3.3%). The ensemble (AIC-weighted
log-logistic) cuts the error roughly in half.

**Notebook 02** (transformations): Log transform + log-logistic gives the
best match to empirical F3/F4 prevalence (total error 0.012). In vivarium,
log transform + lognormal reduces F4 error from +3% to -0.3%.

**Notebook 03** (categorical): Modeling fibrosis as an ordered polytomous
risk factor matches NHANES staging by construction. The trade-off is
losing within-category variation and continuous dose-response capability.